<a href="https://colab.research.google.com/github/rahulgam187/MyLearnings/blob/master/PostProduction_Sounds_BGM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Nuke the fake, temporary imposter folder blocking the way
!rm -rf /content/drive

# 2. Recreate an empty parking spot for your Drive
!mkdir /content/drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 1. Install Audio and Video editing tools
!pip install gTTS audiocraft moviepy pydub librosa==0.8.1

# 2. Clone the Lip-Sync AI
!git clone https://github.com/Rudrabha/Wav2Lip.git

# 3. Download the Wav2Lip weights (The lip-syncing brain)
!wget "https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth" -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth

# 4. Download the Face Detection weights (So it finds the anime mouth)
!wget "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth

print("Setup Complete! Ready for Post-Production.")

In [ ]:
# 1. Install system dependencies for 'av' and video processing
!apt-get update && apt-get install -y libavformat-dev libavdevice-dev ffmpeg

# 2. Downgrade Numpy and install background tools
!pip install "numpy<2.0.0" einops flashy hydra-core hydra_colorlog julius num2words spacy==3.7.6

# 3. Explicitly install a compatible 'av' version
!pip uninstall -y av
!pip install av==11.0.0

# 4. Install Audiocraft
!pip install git+https://github.com/facebookresearch/audiocraft.git

# 5. Install Post-Production tools
!pip install gTTS pydub moviepy

In [ ]:
!pip install --no-deps git+https://github.com/facebookresearch/audiocraft.git

In [ ]:
!pip install xformers
!pip install demucs encodec pesq pystoi torchdiffeq torchmetrics
!pip install --upgrade "click>=8.2.1" typer

In [ ]:
import os

print("--- DIAGNOSTIC CHECK ---")
drive_path = '/content/drive/MyDrive/'
anime_path = '/content/drive/MyDrive/AnimeTest/'

if not os.path.exists(drive_path):
    print("❌ ERROR: Your Google Drive is STILL not mounted. The folder is completely missing.")
else:
    print("✅ Google Drive is successfully mounted!")

    if not os.path.exists(anime_path):
        print("❌ ERROR: Cannot find the 'AnimeTest' folder. Did you spell it differently? Here is what is in your Drive:")
        # Print the first 10 folders in their Drive to see what they are actually named
        print(os.listdir(drive_path)[:10])
    else:
        print("✅ 'AnimeTest' folder found! Here are the exact files inside it:")
        files = os.listdir(anime_path)
        for f in files:
            print(f"   - {f}")

In [ ]:
# 1. Move into the Wav2Lip folder
%cd /content/Wav2Lip

# 2. Run the lip-sync engine loudly so we can see its error
!python inference.py \
  --checkpoint_path checkpoints/wav2lip_gan.pth \
  --face "/content/drive/MyDrive/AnimeTest/scene_00013.mp4" \
  --audio "/content/drive/MyDrive/AnimeTest/Final_Rendered/scene_00013_voice.mp3" \
  --outfile "/content/drive/MyDrive/AnimeTest/Final_Rendered/scene_00013_talking.mp4"

In [ ]:
# Fix the incompatible code in Wav2Lip's audio handler
!sed -i 's/return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,/return librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft, n_mels=hp.num_mels,/' /content/Wav2Lip/audio.py

print("✅ Patch applied! Wav2Lip is now compatible with the new librosa.")

In [ ]:
import pandas as pd
import os
import torch
from gtts import gTTS
from audiocraft.models import MusicGen
import torchaudio
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# --- 1. SETUP PATHS ---
CSV_PATH = '/content/drive/MyDrive/AnimeTest/test_prompts.csv'
VIDEO_DIR = '/content/drive/MyDrive/AnimeTest/'
OUTPUT_DIR = '/content/drive/MyDrive/AnimeTest/Final_Rendered/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. LOAD MUSIC AI ---
print("Loading Music AI. This takes a few seconds...")
model = MusicGen.get_pretrained('facebook/musicgen-small')
model.set_generation_params(duration=5)

# --- 3. READ CSV & START LOOP ---
df = pd.read_csv(CSV_PATH)

for index, row in df.iterrows():
    scene_id = str(row['Scene_ID']).strip() # This will be 'scene_00013'

    # Text to generate
    voice_text = row.get('Voice_Prompt', "Hello! I am testing the audio sync.")
    bgm_prompt = row.get('BGM_Prompt', "lofi chill anime beats, piano")

    # File Paths (Fixed to perfectly match your setup)
    silent_video_path = f"{VIDEO_DIR}{scene_id}.mp4"
    voice_path = f"{OUTPUT_DIR}{scene_id}_voice.mp3"
    bgm_path = f"{OUTPUT_DIR}{scene_id}_bgm.wav"
    talking_video_path = f"{OUTPUT_DIR}{scene_id}_talking.mp4"
    final_path = f"{OUTPUT_DIR}{scene_id}_FINAL.mp4"

    # Strict check: If it's not in the folder exactly as named in the CSV, skip it.
    if not os.path.exists(silent_video_path):
        print(f"[!] Could not find silent video: {silent_video_path}. Skipping row.")
        continue

    print(f"\n[+] Processing {scene_id}...")

    # A. Generate Voice
    print("    -> Generating Voice...")
    tts = gTTS(text=voice_text, lang='en')
    tts.save(voice_path)

    # B. Generate BGM
    print("    -> Generating BGM...")
    wav = model.generate([bgm_prompt])
    torchaudio.save(bgm_path, wav[0].cpu(), 32000)

    # C. Lip-Sync
    print("    -> Running Lip-Sync Engine (Wav2Lip)...")
    os.chdir('/content/Wav2Lip') # Must run from inside this folder

    # Suppressing Wav2Lip's massive console output so your Colab stays clean
    command = f"python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face {silent_video_path} --audio {voice_path} --outfile {talking_video_path}"
    os.system(command)

    # D. Final Mix
    print("    -> Mixing Final Audio (Voice + Background Music)...")
    try:
        video_clip = VideoFileClip(talking_video_path)
        bgm_clip = AudioFileClip(bgm_path).volumex(0.3) # 30% volume for BGM

        # Cut music to match the video's exact length
        bgm_clip = bgm_clip.subclip(0, video_clip.duration)

        # Layer the talking audio and BGM together
        final_audio = CompositeAudioClip([video_clip.audio, bgm_clip])
        video_clip.audio = final_audio

        video_clip.write_videofile(final_path, codec="libx264", audio_codec="aac", logger=None)
        print(f"[SUCCESS] Final Masterpiece saved to: {final_path}")

    except Exception as e:
        print(f"[!] Error during audio mixing for {scene_id}: {e}")

# --- 4. CLEANUP ---
del model
torch.cuda.empty_cache()
print("\nAll Post-Production Complete!")

In [ ]:
# Force Wav2Lip to ignore 'NoneType' face errors and proceed anyway
!sed -i 's/if rects is None:/if False:/g' /content/Wav2Lip/inference.py
!sed -i 's/if len(rects) == 0:/if False:/g' /content/Wav2Lip/inference.py

# A second patch to ensure it doesn't crash when it can't find 'landmarks'
!sed -i 's/raise ValueError("Face not detected")/print("Ignoring missing face...")/g' /content/Wav2Lip/inference.py

print("✅ Wav2Lip has been forced into 'Anime Mode'. It will now process anything you feed it.")

In [ ]:
%cd /content/Wav2Lip
!python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth \
 --face "/content/drive/MyDrive/AnimeTest/scene_00013.mp4" \
 --audio "/content/drive/MyDrive/AnimeTest/Final_Rendered/scene_00013_voice.mp3" \
 --outfile "/content/drive/MyDrive/AnimeTest/Final_Rendered/scene_00013_talking.mp4" \
 --box 0 0 -1 -1 --nosmooth

In [ ]:
# This ensures any previous "box" patches are clean
!sed -i 's/args.box/\[0, 0, -1, -1\] if args.box == \[0, 0, -1, -1\] else args.box/g' /content/Wav2Lip/inference.py

In [ ]:
import pandas as pd
import os
import torch
from gtts import gTTS
from audiocraft.models import MusicGen
import torchaudio
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# --- SETTINGS ---
CSV_PATH = '/content/drive/MyDrive/AnimeTest/test_prompts.csv'
VIDEO_DIR = '/content/drive/MyDrive/AnimeTest/'
OUTPUT_DIR = '/content/drive/MyDrive/AnimeTest/Final_Rendered/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading Music AI...")
model = MusicGen.get_pretrained('facebook/musicgen-small')
model.set_generation_params(duration=5)

df = pd.read_csv(CSV_PATH)

for index, row in df.iterrows():
    scene_id = str(row['Scene_ID']).strip()
    voice_text = row.get('Voice_Prompt', "Testing anime mode.")
    bgm_prompt = row.get('BGM_Prompt', "lofi beats")

    silent_video_path = f"{VIDEO_DIR}{scene_id}.mp4"
    voice_path = f"{OUTPUT_DIR}{scene_id}_voice.mp3"
    bgm_path = f"{OUTPUT_DIR}{scene_id}_bgm.wav"
    talking_video_path = f"{OUTPUT_DIR}{scene_id}_talking.mp4"
    final_path = f"{OUTPUT_DIR}{scene_id}_FINAL.mp4"

    if not os.path.exists(silent_video_path): continue

    print(f"\n[+] FORCING Post-Production for {scene_id}...")

    # A. Audio Generation
    tts = gTTS(text=voice_text, lang='en')
    tts.save(voice_path)
    wav = model.generate([bgm_prompt])
    torchaudio.save(bgm_path, wav[0].cpu(), 32000)

    # B. The "Force" Lip-Sync
    os.chdir('/content/Wav2Lip')

    # We are using --box 0 0 500 500 which covers a 500x500 area from the top left.
    # If your video is 1080p, you can use 0 0 1000 1000.
    import pandas as pd
import os
import torch
from gtts import gTTS
from audiocraft.models import MusicGen
import torchaudio
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip

# --- SETTINGS ---
CSV_PATH = '/content/drive/MyDrive/AnimeTest/test_prompts.csv'
VIDEO_DIR = '/content/drive/MyDrive/AnimeTest/'
OUTPUT_DIR = '/content/drive/MyDrive/AnimeTest/Final_Rendered/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading Music AI...")
model = MusicGen.get_pretrained('facebook/musicgen-small')
model.set_generation_params(duration=5)

df = pd.read_csv(CSV_PATH)

for index, row in df.iterrows():
    scene_id = str(row['Scene_ID']).strip()
    voice_text = row.get('Voice_Prompt', "Testing anime mode.")
    bgm_prompt = row.get('BGM_Prompt', "lofi beats")

    silent_video_path = f"{VIDEO_DIR}{scene_id}.mp4"
    voice_path = f"{OUTPUT_DIR}{scene_id}_voice.mp3"
    bgm_path = f"{OUTPUT_DIR}{scene_id}_bgm.wav"
    talking_video_path = f"{OUTPUT_DIR}{scene_id}_talking.mp4"
    final_path = f"{OUTPUT_DIR}{scene_id}_FINAL.mp4"

    if not os.path.exists(silent_video_path): continue

    print(f"\n[+] FORCING Post-Production for {scene_id}...")

    # A. Audio Generation
    tts = gTTS(text=voice_text, lang='en')
    tts.save(voice_path)
    wav = model.generate([bgm_prompt])
    torchaudio.save(bgm_path, wav[0].cpu(), 32000)

    # B. The "Force" Lip-Sync
    os.chdir('/content/Wav2Lip')

    # We are using --box 0 0 500 500 which covers a 500x500 area from the top left.
    # If your video is 1080p, you can use 0 0 1000 1000.
# --box 0 0 -1 -1 is the "magic" command:
    # It tells Wav2Lip to use the full width and height of whatever video you provide.
    # We use --box 0 0 -1 -1 to force the AI to use the FULL video dimensions.
    # We also add --resize_factor 1 to ensure it doesn't try to shrink the image.
    command = f"python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth " \
              f"--face {silent_video_path} --audio {voice_path} " \
              f"--outfile {talking_video_path} " \
              f"--box 0 0 -1 -1 --nosmooth --resize_factor 1"

    os.system(command)

    # C. Mixing
    if os.path.exists(talking_video_path):
        try:
            video_clip = VideoFileClip(talking_video_path)
            bgm_clip = AudioFileClip(bgm_path).volumex(0.3).subclip(0, video_clip.duration)
            video_clip.audio = CompositeAudioClip([video_clip.audio, bgm_clip])
            video_clip.write_videofile(final_path, codec="libx264", audio_codec="aac", logger=None)
            print(f"✅ SUCCESS: {final_path}")
        except Exception as e:
            print(f"❌ Mix Error: {e}")
    else:
        print(f"❌ Still failed. This means the Wav2Lip engine is crashing internally.")

del model
torch.cuda.empty_cache()

    # C. Mixing
if os.path.exists(talking_video_path):
        try:
            video_clip = VideoFileClip(talking_video_path)
            bgm_clip = AudioFileClip(bgm_path).volumex(0.3).subclip(0, video_clip.duration)
            video_clip.audio = CompositeAudioClip([video_clip.audio, bgm_clip])
            video_clip.write_videofile(final_path, codec="libx264", audio_codec="aac", logger=None)
            print(f"✅ SUCCESS: {final_path}")
        except Exception as e:
            print(f"❌ Mix Error: {e}")
        else:
            print(f"❌ Still failed. This means the Wav2Lip engine is crashing internally.")

try:
    del model
    torch.cuda.empty_cache()
except NameError:
    pass

In [ ]:
import pandas as pd

# Define the 5 cohesive scenes
data = {
    'Scene_ID': ['scene_00015', 'scene_00016', 'scene_00017', 'scene_00018', 'scene_00019'],
    'Voice_Prompt': [
        "The morning air is so crisp today. I feel like something wonderful is about to happen.",
        "Wait... did you hear that? I think there's someone else in the forest with us.",
        "Oh! It was just a little fox. You gave me quite a scare, didn't you, little one?",
        "Look at how the sky is changing colors. The sunset looks like a painting come to life.",
        "I'm glad we spent the day together. Let's do this again tomorrow, okay?"
    ],
    'BGM_Prompt': [
        "peaceful acoustic guitar morning vibes",
        "mysterious ambient synth low tension",
        "playful woodwind melody lighthearted",
        "lofi hip hop warm sunset chill",
        "soft piano emotional ending"
    ]
}

df = pd.DataFrame(data)

# Save to your Google Drive path
csv_export_path = '/content/drive/MyDrive/AnimeTest/test_prompts.csv'
df.to_csv(csv_export_path, index=False)

print(f"✅ New 5-scene CSV saved to: {csv_export_path}")